# Estructura del dataset

- Session 1: Neutral, Smile
- Session 2: Neutral, surprise, squint
- Session 3: Neutral, smile, disgust
- Session 4: Neutral, Scream, Neutral


In [2]:
import pandas as pd

df = pd.read_csv("/home12TB1/database/recognition/faces/MultiPie/demographic_info_cropped.csv")
print("Unique recordingids:", df['recording_id'].unique())
print("Unique Session IDs", df['session_id'].unique())

Unique recordingids: [2 1 3]
Unique Session IDs ['session01' 'session02' 'session03' 'session04']


# Estadísticas del dataset

Número de usuarios: `748120`

In [10]:
print("Unique user id", df["subject_id"])
print("Number of cameras: ", len(df["camera_id"].unique()))
print("Pics of user 22:", len(df[df["subject_id"] == 22 ])/15)

Unique user id 0         155
1         155
2         155
3         155
4         155
         ... 
748115    342
748116    342
748117    342
748118    342
748119    342
Name: subject_id, Length: 748120, dtype: int64
Number of cameras:  15
Pics of user 22: 218.93333333333334


In [ ]:
print(df[df["subject_id"] == 22].groupby(['session_id', 'recording_id']).size())

session_id  recording_id
session01   1               300
            2               300
session02   1               300
            2               300
            3               300
session03   1               297
            2               300
            3               287
session04   1               300
            2               300
            3               300
dtype: int64


# Número de hombres y mujeres

In [14]:
print("Numero de hombres", len(df[df["gender"] == "Male"]))
print("Numero de mujeres", len(df[df["gender"] == "Female"]))


Numero de hombres 535793
Numero de mujeres 212327


# Numero de hombres y mujeres para las 6 expresiones

In [6]:
import pandas as pd

# 1. Cargar el CSV
df = pd.read_csv("/home12TB1/database/recognition/faces/MultiPie/demographic_info_cropped.csv")

# 2. Función de mapeo corregida
def map_expression(row):
    # Aseguramos que tratamos con strings y enteros limpios
    s = str(row['session_id']).lower() # Convertimos a minúsculas por si acaso
    r = int(row['recording_id'])

    if r == 1: return 0 # Neutral
    
    # Usamos "01" o "session01" dependiendo de cómo venga en tu CSV
    if '01' in s and r == 2: return 1 # Smile
    if '02' in s and r == 2: return 2 # Surprise
    if '02' in s and r == 3: return 3 # Squint
    if '03' in s and r == 2: return 1 # Smile
    if '03' in s and r == 3: return 4 # Disgust
    if '04' in s and r == 2: return 5 # Scream
    if '04' in s and r == 3: return 0 # Neutral
    return -1

# 3. Aplicar
df['expression'] = df.apply(map_expression, axis=1)

# 4. Filtrar y limpiar
df_valid = df[df['expression'] != -1].copy()

# 5. RESULTADOS
print("--- Tabla de Contingencia ---")
# Añadimos margins=True para ver los totales de fila y columna de un vistazo
contingency_table = pd.crosstab(df_valid['expression'], df_valid['gender'], margins=True)
print(contingency_table)

print("\n--- Distribución de Expresiones ---")
# Mapeo de nombres para que sea más fácil de leer
exp_names = {0: "Neutral", 1: "Smile", 2: "Surprise", 3: "Squint", 4: "Disgust", 5: "Scream"}
counts = df_valid['expression'].value_counts().sort_index()
for idx, count in counts.items():
    print(f"{exp_names[idx]}: {count}")

--- Tabla de Contingencia ---
gender      Female    Male     All
expression                        
0            98390  246601  344991
1            40667  101992  142659
2            17105   43617   60722
3            16534   43814   60348
4            19177   49123   68300
5            20454   50646   71100
All         212327  535793  748120

--- Distribución de Expresiones ---
Neutral: 344991
Smile: 142659
Surprise: 60722
Squint: 60348
Disgust: 68300
Scream: 71100


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

CSV_PATH = "/home12TB1/database/recognition/faces/MultiPie/demographic_info_cropped.csv"

def generate_labels(df):
    """
    Generates the 'temp_label' column based on session and recording IDs.
    Returns the modified DataFrame with valid labels only.
    """
    def map_row(row):
        session = str(row['session_id'])
        rec = row['recording_id']

        if rec == 1: 
            return 0  # Neutral
        
        # Session-specific logic
        if 'session01' in session:
            if rec == 2: return 1  # Smile
        elif 'session02' in session:
            if rec == 2: return 2  # Surprise
            if rec == 3: return 3  # Squint
        elif 'session03' in session:
            if rec == 2: return 1  # Smile
            if rec == 3: return 4  # Disgust
        elif 'session04' in session:
            if rec == 2: return 5  # Scream
            if rec == 3: return 0  # Neutral
            
        return -1

    df['temp_label'] = df.apply(map_row, axis=1)
    df = df[df['temp_label'] != -1].reset_index(drop=True)
    return df

def analyze_training_data(csv_path):
    print("Loading data...")
    df = pd.read_csv(csv_path)
    
    # 1. Apply the real label generation for all classes
    df = generate_labels(df)

    # 2. Replicate your Subject-Disjoint Split (70% Train)
    subjects_df = df[['subject_id', 'gender']].drop_duplicates()
    
    train_subs, _ = train_test_split(
        subjects_df,
        test_size=0.3,
        stratify=subjects_df['gender'],
        random_state=42
    )

    # 3. Filter the main dataframe for training subjects
    train_df = df[df['subject_id'].isin(train_subs['subject_id'])]

    # 4. Map numeric labels to names for better readability
    label_map = {
        0: 'Neutral', 1: 'Smile', 2: 'Surprise', 
        3: 'Squint', 4: 'Disgust', 5: 'Scream'
    }
    # Create a copy to avoid SettingWithCopyWarning
    train_df = train_df.copy()
    train_df['Expression'] = train_df['temp_label'].map(label_map)

    # 5. Calculate TOTAL IMAGES and UNIQUE IDENTITIES
    print("\n=== TRAINING SET ANALYSIS (70% SPLIT) ===")
    
    # Group by Expression and Gender
    summary = train_df.groupby(['Expression', 'gender']).agg(
        Total_Images=('subject_id', 'count'),
        Unique_Identities=('subject_id', 'nunique')
    ).reset_index()

    # Sort to match the order: Neutral, Smile, Surprise, Squint, Disgust, Scream
    cat_type = pd.CategoricalDtype(categories=list(label_map.values()), ordered=True)
    summary['Expression'] = summary['Expression'].astype(cat_type)
    summary = summary.sort_values(['Expression', 'gender']).reset_index(drop=True)

    print(summary.to_string(index=False))
    
    # Specifically print the Squint bottleneck info
    print("\n" + "="*40)
    print("THE BOTTLENECK (SQUINT - FEMALES)")
    print("="*40)
    squint_f = summary[(summary['Expression'] == 'Squint') & (summary['gender'] == 'Female')]
    if not squint_f.empty:
        images = squint_f['Total_Images'].values[0]
        identities = squint_f['Unique_Identities'].values[0]
        print(f"-> Max female images available (Your N_limit): {images}")
        print(f"-> Max female identities available: {identities}")
    else:
        print("No female squint data found in this split!")

if __name__ == "__main__":
    analyze_training_data(CSV_PATH)

Loading data...

=== TRAINING SET ANALYSIS (70% SPLIT) ===
Expression gender  Total_Images  Unique_Identities
   Neutral Female         86596                278
   Neutral   Male        221834                301
     Smile Female         35436                233
     Smile   Male         91450                283
  Surprise Female         15137                150
  Surprise   Male         39595                183
    Squint Female         14634                143
    Squint   Male         39743                182
   Disgust Female         16748                155
   Disgust   Male         44155                205
    Scream Female         18172                142
    Scream   Male         45492                211

THE BOTTLENECK (SQUINT - FEMALES)
-> Max female images available (Your N_limit): 14634
-> Max female identities available: 143 (This should match Ivan's chart!)


# Comprobación de los csvs generados

In [2]:
import pandas as pd

# fsb_df = pd.read_csv("/home12TB1/database/recognition/faces/affwild2/img2pose_ann/affwild2_train_pose_fsb.csv")
# i2p_df = pd.read_csv("/home12TB1/database/recognition/faces/affwild2/img2pose_ann/affwild2_pose_results_mario.csv")

multipie_df = pd.read_csv("/home12TB1/database/recognition/faces/affectnet/affectnetplus_train_annotations.csv")
print(f"La longitud de affectnet es de {len(multipie_df)}")

# print(f"La longitud del fsb es de {len(fsb_df)}")
# print(f"La longitud del i2p es de {len(i2p_df)}")

La longitud de affectnet es de 287651


In [5]:
import pandas as pd

csv_list = [
    "/home12TB1/database/recognition/faces/affectnet/img2pose_ann/train_pose_fsb.csv",
    "/home12TB1/database/recognition/faces/affectnet/img2pose_ann/val_pose_fsb.csv",
    "/home12TB1/database/recognition/faces/affectnet/img2pose_ann/no_human_annotated_pose_fsb.csv",
    "/home12TB1/database/recognition/faces/RAF-DB/img2pose_ann/train_pose_fsb.csv",
    "/home12TB1/database/recognition/faces/RAF-DB/img2pose_ann/test_pose_fsb.csv"
]

nombres = ['AffectNet Train', 'AffectNet Val', 'AffectNet NHA', 'RAF-DB Train', 'RAF-DB Val']

print(f"{'Dataset':<20} | {'Max FSB':<10} | {'Path del valor más alto'}")
print("-" * 80)

for file, name in zip(csv_list, nombres):
    try:
        df = pd.read_csv(file)
        
        # 1. Encontrar el valor máximo
        max_val = df['FSB'].max()
        
        # 2. Encontrar el path de ese valor máximo
        # Usamos idxmax() para obtener el índice de la fila con el valor más alto
        max_path = df.loc[df['FSB'].idxmax(), 'path']
        
        # 3. Contar cuántos superan el límite de 255
        over_limit = len(df[df['FSB'] > 255])
        
        print(f"{name:<20} | {max_val:>10.2f} | {max_path}")
        if over_limit > 0:
            print(f"   --> ¡OJO! Hay {over_limit} filas que superan el valor de 255.")
            
    except Exception as e:
        print(f"Error en {name}: {e}")

Dataset              | Max FSB    | Path del valor más alto
--------------------------------------------------------------------------------
AffectNet Train      |     254.19 | //home12TB1/database/recognition/faces/affectnet/human_annotated/train_set/images/91432.jpg
AffectNet Val        |     252.87 | //home12TB1/database/recognition/faces/affectnet/human_annotated/validation_set/images/2682.jpg
AffectNet NHA        |     254.47 | /home12TB1/database/recognition/faces/affectnet/no_human_annotated/images/35766.jpg
RAF-DB Train         |     244.68 | /home12TB1/database/recognition/faces/RAF-DB/Image/aligned/train_04762_aligned.jpg
RAF-DB Val           |     240.84 | /home12TB1/database/recognition/faces/RAF-DB/Image/aligned/test_2515_aligned.jpg


In [5]:
import pandas as pd
affnectnet_df  = pd.read_csv("/home12TB1/database/recognition/faces/affectnet/affectnetplus_train_annotations.csv")
affectnet_test_df = pd.read_csv("/home12TB1/database/recognition/faces/affectnet/affectnetplus_test_annotations_quality_illum.csv")
print(affnectnet_df['human_label'].value_counts().min())
print(affectnet_test_df['human_label'].value_counts())

3750
human_label
7    500
3    500
4    500
5    500
6    500
2    500
1    500
0    499
Name: count, dtype: int64


In [2]:
# Estudiar anotaciones Affectnet human_label vs soft

import pandas as pd
import numpy as np

df = pd.read_csv("/home12TB1/database/recognition/faces/affectnet/affectnetplus_train_annotations.csv")

soft_cols = [f'soft_{i}' for i in range(8)]

# 3. Obtener la predicción del "modelo soft" (la clase con mayor probabilidad)
# idxmax devuelve el nombre de la columna con el valor máximo
df['soft_prediction'] = df[soft_cols].idxmax(axis=1).str.replace('soft_', '').astype(int)

# 4. Calcular la coincidencia
df['match'] = df['human_label'] == df['soft_prediction']

# 5. Agrupar por emoción humana para ver el desastre por clases
report = df.groupby('human_label').agg(
    total_images=('match', 'count'),
    matches=('match', 'sum'),
    accuracy_vs_soft=('match', 'mean')
).reset_index()

# Mapeo de nombres para que se entienda mejor
emotions = {0: 'Neutral', 1: 'Happy', 2: 'Sad', 3: 'Surprise', 
            4: 'Fear', 5: 'Disgust', 6: 'Anger', 7: 'Contempt'}
report['emotion_name'] = report['human_label'].map(emotions)

print("--- REPORTE DE COINCIDENCIA: HUMANO vs SOFT LABELS ---")
print(report[['human_label', 'emotion_name', 'total_images', 'accuracy_vs_soft']])

# 6. Bonus: ¿A qué clase se va el error? (Matriz de confusión de etiquetas)
print("\n--- MATRIZ DE DISCREPANCIA (Filas: Humano, Columnas: Soft) ---")
discrepancy_matrix = pd.crosstab(df['human_label'], df['soft_prediction'], normalize='index')
print(discrepancy_matrix)

--- REPORTE DE COINCIDENCIA: HUMANO vs SOFT LABELS ---
   human_label emotion_name  total_images  accuracy_vs_soft
0            0      Neutral         74874          0.686874
1            1        Happy        134415          0.862508
2            2          Sad         25459          0.320987
3            3     Surprise         14090          0.348758
4            4         Fear          6378          0.266071
5            5      Disgust          3803          0.259532
6            6        Anger         24882          0.428020
7            7     Contempt          3750          0.127200

--- MATRIZ DE DISCREPANCIA (Filas: Humano, Columnas: Soft) ---
soft_prediction         0         1         2         3         4         5  \
human_label                                                                   
0                0.686874  0.101357  0.050966  0.031052  0.011045  0.020314   
1                0.065930  0.862508  0.011628  0.008593  0.003363  0.011435   
2                0.326329